In [9]:
import json


with open("../../Experiment/SystemEvaluation/Shelf/shelf_single_no_color.json", "r", encoding="utf-8") as f:
    tasks = json.load(f)

with open("../../Experiment/SystemEvaluation/Shelf/Shelf.json", "r", encoding="utf-8") as f:
    shelf_data = json.load(f)



print("===============Shelf Datas==============" )
print(shelf_data[:3])

print("===============Tasks==============" )
print(tasks[:3])



===============Shelf Datas==============
[{'id': '409f6dff-d2e2-43c8-8ba5-2492b3e2e5d6', 'name': 'Shelf Light 1', 'position': {'x': -2.01285267, 'y': 0.429263115, 'z': 3.560356}, 'distance_from_user': 4.112417, 'eye_centrality_score': 30.5167942}, {'id': '80e64f7b-0589-4de7-977a-9d189a66fd1d', 'name': 'Shelf Light 4', 'position': {'x': -0.02322777, 'y': 0.447263241, 'z': 3.40095115}, 'distance_from_user': 3.43031359, 'eye_centrality_score': 8.679442}, {'id': '7ad8857c-4547-4f42-bcc0-dc17bd8e6fee', 'name': 'Shelf Light 7', 'position': {'x': 1.37828112, 'y': 0.429263115, 'z': 3.28866482}, 'distance_from_user': 3.5915513, 'eye_centrality_score': 24.4718132}]
===============Tasks==============
[{'command': '一番左上のライトをつけて', 'target_type': 'single', 'has_color': False, 'target_devices': [{'device_name': 'Shelf Light 1', 'color': 'white'}]}, {'command': '右下のライトを点灯して', 'target_type': 'single', 'has_color': False, 'target_devices': [{'device_name': 'Shelf Light 9', 'color': 'white'}]}, {'command

In [13]:

from no_tool_agent_runner import getSpatialRunnerForEvaluation
from sr_app_types.no_tool_agent_types import State, FilterAgentType, FilterAgentOutput
from langchain_core.messages import ToolMessage, HumanMessage, SystemMessage



def build_state_from_task(task, shelf_data):
    command = task["command"]
    
    tool_selection = FilterAgentOutput(
        filter_type="no_filter",
        params={"description": "No filter applied, but candidate devices are provided."},
        reasoning="This is for evaluating spatial reasoning performance without filtering."
    )

    return State(
        user_prompt=command,
        filterAgent=FilterAgentType(
            devices=shelf_data,
            output_tool_selection=tool_selection
        )
    )
def evaluate_task(task, result, shelf_data):
    # id→nameマップを構築
    id_to_name = {d["id"]: d["name"] for d in shelf_data}
    pred_ids = {d["id"] for d in result["agent_output"]["devices"]}
    pred_names = {id_to_name[pid] for pid in pred_ids if pid in id_to_name}
    gt_names = {d["device_name"] for d in task["target_devices"]}
    return pred_names == gt_names
evaluation_results = {
    "total": len(tasks),
    "correct": 0,
    "accuracy": 0.0,
    "results": []  # 個別タスクの結果
}

runner = getSpatialRunnerForEvaluation()

for i, task in enumerate(tasks):
    state = build_state_from_task(task, shelf_data)
    result = runner.invoke(state)

    # 評価判定
    is_correct = evaluate_task(task, result, shelf_data)

    # id→nameマップで出力変換
    id_to_name = {d["id"]: d["name"] for d in shelf_data}
    pred_ids = {d["id"] for d in result["agent_output"]["devices"]}
    pred_names = {id_to_name[pid] for pid in pred_ids if pid in id_to_name}
    gt_names = {d["device_name"] for d in task["target_devices"]}

    # 結果格納
    evaluation_results["results"].append({
        "task_index": i,
        "command": task["command"],
        "predicted_names": sorted(pred_names),
        "ground_truth_names": sorted(gt_names),
        "is_correct": is_correct,
        "agent_output": result["agent_output"],
    })

    if is_correct:
        evaluation_results["correct"] += 1
        print(f"[✅] Task {i}: {task['command']}")
    else:
        print(f"[❌] Task {i}: {task['command']}")
        print(f"     🔸 Predicted: {sorted(pred_names)}")
        print(f"     🔹 Expected : {sorted(gt_names)}")

# Accuracy計算
evaluation_results["accuracy"] = evaluation_results["correct"] / evaluation_results["total"]

print(f"\n=== 📊 Accuracy: {evaluation_results['accuracy']:.2%} ({evaluation_results['correct']}/{evaluation_results['total']}) ===")


==========[SR AGENT NODE]=========
OUTPUT:  デバイスの位置を比較し、x軸で最も左かつy軸で最も高い位置にあるデバイスを選定しました。Shelf Light 1が最も左上に位置しているため、このデバイスを操作対象としました。
RESPONSE:  一番左上のライトをつけました。
[SR AGENT TIME ELAPSED] 3.6086 sec
[✅] Task 0: 一番左上のライトをつけて
==========[SR AGENT NODE]=========
OUTPUT:  『右下』という指示に基づき、x軸が正の方向、y軸が負の方向に位置するライトを選択しました。Shelf Light 9はxが最も大きく、yが最も小さいため、空間的に右下に位置していると判断されました。
RESPONSE:  右下のライトを点灯しました。
[SR AGENT TIME ELAPSED] 3.0885 sec
[✅] Task 1: 右下のライトを点灯して
==========[SR AGENT NODE]=========
OUTPUT:  デバイスはy座標に基づいて3つの段に分類されました。中央の段はy ≈ 0.44の範囲にあり、その中でeye_centrality_scoreが最も低いデバイスを中央として選択しました。
RESPONSE:  真ん中の段の中央のデバイスをオンにしました。
[SR AGENT TIME ELAPSED] 2.9635 sec
[❌] Task 2: 真ん中の段の中央のやつをオンにして
     🔸 Predicted: ['Shelf Light 4']
     🔹 Expected : ['Shelf Light 5']
==========[SR AGENT NODE]=========
OUTPUT:  与えられたデバイスの中から、y座標が最も高いものを上の段と判断しました。上の段に属するデバイスの中で、x座標が最も大きいものを右端のライトとして選択しました。
RESPONSE:  上の段の右端のライトをオンにしました。
[SR AGENT TIME ELAPSED] 3.3300 sec
[✅] Task 3: 上の段の右端のライトをお願い
==========[SR AGENT NODE]

In [18]:
for result in evaluation_results["results"]:
    if not result["is_correct"]:
        print(result["task_index"], result["command"],  "\n")

2 真ん中の段の中央のやつをオンにして 

7 中央上のライトをつけて 

8 棚の中央にあるライトを点灯して 

10 棚の右列真ん中のやつを点灯 

13 下段中央を点灯して 

14 真ん中の棚の右側をつけて 



In [2]:

from dotenv import load_dotenv
load_dotenv("../.env")
from no_tool_agent_runner import getSpatialRunner
from sr_app_types.no_tool_agent_types import State, FilterAgentType, FilterAgentOutput
from langchain_core.messages import ToolMessage, HumanMessage, SystemMessage

runner = getSpatialRunner()
task_index = 2

command_prompt = tasks[task_index]["command"]
tool_selection = FilterAgentOutput(
    filter_type="none",
    params={"description": "No filter applied, but candidate devices are provided."},
    reasoning="No filter used — direct device reasoning"
)


input_state: State = State(
        user_prompt=command_prompt,
        filterAgent=FilterAgentType(devices=shelf_data, output_tool_selection=tool_selection)

    )
res = runner.invoke(input_state)
# ID → 名前マップを作成（shelf_dataに基づく）
id_to_name = {device["id"]: device["name"] for device in shelf_data}

# LLMが選択したデバイスのID
predicted_ids = {d["id"] for d in res["agent_output"]["devices"]}

# それに対応する名前
predicted_names = {id_to_name[pid] for pid in predicted_ids if pid in id_to_name}

# 正解（target）デバイス名
target_names = {d["device_name"] for d in tasks[task_index]["target_devices"]}

# 比較して出力
is_correct = predicted_names == target_names

print("=== コマンド ===")
print(command_prompt)
print("=== LLMが選んだデバイス ===")
print(predicted_names)
print("=== 正解デバイス ===")
print(target_names)
print("=== 判定結果 ===")
print("✅ 正解" if is_correct else "❌ 不正解")

==========[SR AGENT NODE]=========
OUTPUT:  The user requested to turn on the 'center' lights. Spatially, devices were grouped based on their x and y positions. Shelf Light 4 (x ≈ 0, y ≈ 0.45) and Shelf Light 5 (x ≈ 0.53, y ≈ -0.34) are closest to the center in both axes, forming a central vertical band. Other devices are further left/right or higher/lower, so only these two are considered 'center' lights.
RESPONSE:  Turned on the lights located at the center.
[SR AGENT TIME ELAPSED] 5.3155 sec

=====================[OPERATOR TOOL] operateDevice=====================
Devices to operate:  [DeviceControlData(id='80e64f7b-0589-4de7-977a-9d189a66fd1d', state=True, intensity=100, color=RGBColor(r=255, g=255, b=255)), DeviceControlData(id='f2505762-f11e-446b-a340-90bbfae0460b', state=True, intensity=100, color=RGBColor(r=255, g=255, b=255))]
Sending Operate Request to Test Server.
ERROR OCCURRED DURING OPERATION TOOL:  [WinError 10061] 対象のコンピューターによって拒否されたため、接続できませんでした。
=== コマンド ===
真ん中の電気をオンに

In [18]:
# ID → 名前マップを作成（shelf_dataに基づく）
id_to_name = {device["id"]: device["name"] for device in shelf_data}

# LLMが選択したデバイスのID
predicted_ids = {d["id"] for d in res["agent_output"]["devices"]}

# それに対応する名前
predicted_names = {id_to_name[pid] for pid in predicted_ids if pid in id_to_name}

# 正解（target）デバイス名
target_names = {d["device_name"] for d in tasks[0]["target_devices"]}

# 比較して出力
is_correct = predicted_names == target_names

print("=== コマンド ===")
print(command_prompt)
print("=== LLMが選んだデバイス ===")
print(predicted_names)
print("=== 正解デバイス ===")
print(target_names)
print("=== 判定結果 ===")
print("✅ 正解" if is_correct else "❌ 不正解")


=== コマンド ===
一番左上のライトをつけて
=== LLMが選んだデバイス ===
{'Shelf Light 1'}
=== 正解デバイス ===
{'Shelf Light 1'}
=== 判定結果 ===
✅ 正解


In [3]:
def evaluate_task(task, result, shelf_data):
    id_to_name = {d["id"]: d["name"] for d in shelf_data}
    id_to_color = {d["id"]: d.get("color", "white") for d in shelf_data}

    # 出力から name/color を抽出
    predicted = result["agent_output"]["devices"]

    if task["target_type"] == "single":
        # 単一（colorあり/なし）の比較
        gt = task["target_devices"][0]
        pred = predicted[0] if predicted else {}
        correct_name = id_to_name.get(pred.get("id")) == gt["device_name"]
        correct_color = pred.get("color", "white") == gt.get("color", "white")
        return correct_name and (not task["has_color"] or correct_color)
    
    else:  # multi
        gt_set = set((d["device_name"], d["color"]) if task["has_color"] else d["device_name"] for d in task["target_devices"])
        pred_set = set(
            (id_to_name.get(d["id"]), d.get("color", "white")) if task["has_color"] else id_to_name.get(d["id"])
            for d in predicted if d["id"] in id_to_name
        )
        return gt_set == pred_set


In [10]:
import os
print(os.getcwd())

c:\Users\Tenma\Desktop\Keio\InteractiveSmartHome\InteractiveSmartHome\LLMServer\workspace


In [34]:
import json
from pathlib import Path
import pandas as pd

# ファイルパスの定義
base_path = Path("C:/Users/Tenma/Desktop/Keio/InteractiveSmartHome/InteractiveSmartHome/Experiment/SystemEvaluation/Shelf")
file_paths = {
    "shelf_single_no_color": base_path / "shelf_single_no_color.json",
    "shelf_single_with_color": base_path / "shelf_single_color.json",
    "shelf_multi_no_color": base_path / "shelf_multi_no_color.json",
    "shelf_multi_with_color": base_path / "shelf_multi_with_color.json"
}

# 各ファイルの全データをロード
examples = {}
for category, path in file_paths.items():
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
        # commandを含む正しいフォーマットのみに限定
        examples[category] = [task for task in data if "command" in task][:5]

# 表形式に整理
summary_data = []
for category, tasks in examples.items():
    for task in tasks:
        summary_data.append({
            "category": category,
            "command": task["command"],
            "target_type": task["target_type"],
            "has_color": task.get("has_color", False),
            "device_count": len(task["target_devices"]),
            "devices": ", ".join([
                f"{d['device_name']}({d.get('color', '-')})"
                for d in task["target_devices"]
            ])
        })

df = pd.DataFrame(summary_data)



In [36]:
df

,category,command,target_type,has_color,device_count,devices
0,shelf_single_no_color,一番左上のライトをつけて,single,False,1,Shelf Light 1(white)
1,shelf_single_no_color,右下のライトを点灯して,single,False,1,Shelf Light 9(white)
2,shelf_single_no_color,真ん中の電気をオンにして,single,False,1,Shelf Light 5(white)
3,shelf_single_no_color,上の段の右端のライトをお願い,single,False,1,Shelf Light 7(white)
4,shelf_single_no_color,真ん中の下のライトを点けて,single,False,1,Shelf Light 6(white)
5,shelf_single_with_color,一番左上のライトを青にしてつけて,single,True,1,Shelf Light 1(青)
6,shelf_single_with_color,右下のライトを緑にして点灯して,single,True,1,Shelf Light 9(緑)
7,shelf_single_with_color,真ん中の段の中央のやつをシアンにしてオンにして,single,True,1,Shelf Light 5(シアン)
8,shelf_single_with_color,上の段の右端のライトをピンクにしてお願い,single,True,1,Shelf Light 7(ピンク)
9,shelf_single_with_color,中央列の一番下のライトを紫にして点けて,single,True,1,Shelf Light 6(紫)


In [37]:
def is_correct_single_no_color(predicted, ground_truth):
    return len(predicted) == 1 and len(ground_truth) == 1 and \
           predicted[0]["device_name"] == ground_truth[0]["device_name"]
def is_correct_single_with_color(predicted, ground_truth):
    return len(predicted) == 1 and len(ground_truth) == 1 and \
           predicted[0]["device_name"] == ground_truth[0]["device_name"] and \
           predicted[0]["color"] == ground_truth[0]["color"]
def is_correct_multi_no_color(predicted, ground_truth):
    pred_names = {d["device_name"] for d in predicted}
    gt_names = {d["device_name"] for d in ground_truth}
    return pred_names == gt_names

def is_correct_multi_with_color(predicted, ground_truth):
    pred_pairs = {(d["device_name"], d["color"]) for d in predicted}
    gt_pairs = {(d["device_name"], d["color"]) for d in ground_truth}
    return pred_pairs == gt_pairs


In [38]:
for task in tasks:
    pred = result["agent_output"]["devices"]
    gt = task["target_devices"]

    if task["target_type"] == "single" and not task["has_color"]:
        is_correct = is_correct_single_no_color(pred, gt)
    elif task["target_type"] == "single" and task["has_color"]:
        is_correct = is_correct_single_with_color(pred, gt)
    elif task["target_type"] == "multi" and not task["has_color"]:
        is_correct = is_correct_multi_no_color(pred, gt)
    elif task["target_type"] == "multi" and task["has_color"]:
        is_correct = is_correct_multi_with_color(pred, gt)


NameError: name 'result' is not defined